# Part 10, notebook 1 of 5: three ways to spend model contexts

Companion to *When One Agent Is Not Enough*. The five notebooks follow the post in order. Each explains the idea with a daily-life picture, shows the real source, and runs it. **Cells marked "Your turn" are for you to change and rerun.**

| # | Notebook | Post section | Needs the model server? |
|---|---|---|---|
| 10a | `10a_three_ways` | Three ways to spend model contexts | two short live cells |
| 10b | `10b_supervisor_and_handoff` | A supervisor and two specialists | live cells (about 1 minute) |
| 10c | `10c_fanout_and_panel` | One sub-agent per box, and a panel | live cells (about 30 seconds) |
| 10d | `10d_the_200_image_results` | The results | no (reads the saved runs) |
| 10e | `10e_failures_and_tests` | What goes wrong | no |

**The picture.** A small clinic. Option A: one doctor does everything, from the first question to the last prescription, and keeps
the whole story in their head. Option B: a front-desk manager who cannot examine anyone passes written notes to a nurse (who measures)
and a doctor (who decides). Option C: every sample goes to its own technician, who sees only that sample, and a clerk collects the answers.

In AI words: a **context** is the room in which one model can see and remember things (its prompt plus everything said so far).
A **sub-agent** is another model call, with its own room, that does one job and sends back only its answer.

In [1]:
import sys, json, os
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve()))     # this notebook lives in literate/, the code in notebooks/agentic-zero-to-advanced/
OUT = Path("..").resolve() / "out"
os.environ.setdefault("PART10_MODEL", "qwen3.8-27b")   # the model name the local vLLM server serves
import sub_agents as sa
import sub_summary as ss
%matplotlib inline
d = ss.load()
print("arms with saved runs:", {arm: sorted(runs) for arm, runs in d.items()})

arms with saved runs: {'single': [0], 'batched': [0, 1, 2], 'fanout': [0, 1, 2], 'panel3': [0, 1], 'supervisor': [0]}


## What the dataset allows

The plan for Part 10 said "sub-agents per defect category or per inspection stage". This dataset has **one** defect category (voids),
so a per-category split cannot be tested honestly. What can be split: by **stage** (measure, then judge) and by **box** (one judge per candidate).
The saved runs show how many boxes an image has. That number is what makes a context grow.

In [2]:
from collections import Counter
recs = list(d["batched"][0].values())
hist = Counter(min(r["n_candidates"], 8) for r in recs)
for k in sorted(hist):
    print(f"{'8+' if k == 8 else k:>3} candidate boxes: {hist[k]:3d} images")
print("total boxes:", sum(r["n_candidates"] for r in recs))

  0 candidate boxes:  55 images
  1 candidate boxes:  36 images
  2 candidate boxes:  29 images
  3 candidate boxes:  26 images
  4 candidate boxes:  20 images
  5 candidate boxes:   6 images
  6 candidate boxes:  11 images
  7 candidate boxes:   5 images
 8+ candidate boxes:  12 images
total boxes: 513


## Run one image through two designs, live

`arm_single` is Part 5's agent with all tools in one context. `arm_fanout` finds and measures with plain code, then gives each box to a
sub-agent that sees **only that box**. Both return the same record shape, with every model call counted by the `Meter`.

**Your turn:** change `IMAGE` to any `test_00000` to `test_00199` and rerun. Images with more candidate boxes show the difference best (try `test_00046`).

In [3]:
IMAGE = "test_00046"

rows = []
for name in ("single", "fanout"):
    r = sa.ARMS[name](IMAGE)
    rows.append(r)
    print(f"{name:7s} boxes={r['n_candidates']} kept={r['kept']}  tp/fp/fn={r['tp']}/{r['fp']}/{r['fn']}  "
          f"model calls={r['calls']}  tokens={r['prompt_tokens'] + r['completion_tokens']}  biggest context={r['peak_prompt']} tokens  {r['seconds']} s")

single  boxes=4 kept=[0]  tp/fp/fn=1/0/1  model calls=3  tokens=3499  biggest context=1376 tokens  26.4 s


fanout  boxes=4 kept=[0]  tp/fp/fn=1/0/1  model calls=4  tokens=1250  biggest context=259 tokens  5.1 s


**What to look at:** `biggest context`. The single agent's room grows with every tool result it keeps. Each fan-out judge's room is the
same small size whatever the image holds. That is the whole mechanism; the post measures whether it matters (here, not much: both are tiny next to the 65,536-token window).